Bing Chen

ST 554 Project 3

4/19/2026

# Introduction

For this project, we want to fit a machine learning model, read in a stream of data created ourselves, and use the model to do predictions on the stream and write those out to the console. 

First, let's read in the data we want to fit the model on. This data contains information about power
consumption from different zones of Tetouan city and various variables like time of day, temperature, and
humidity. We'll read in using `pandas`, then convert into a Spark dataframe. 

In [1]:
import pandas as pd 

power = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")

power.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


Now let's convert it into a Spark dataframe. We need to create a spark session and then convert. 

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master('local[*]').appName('my_app').getOrCreate()

spower = spark.createDataFrame(power)


Now we have the data ready, we want to make prediction on `Power_Zone_3`, and we'll use all other variables as the predictor. We want to fit an elastic net model using CV. Before we fit the model, let's consider some transformations. 

Ler's check if the `Hour` column is stored as double, if not we want to use `SQLTransformer` to cast the variable as a double Then, we'll `Binarize` the Hour column based on the column being less than 6.5 or not to separate days and nights. We'll consider `One-hot encode` the Month column, run a PCA fit on the `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows columns`. To do this, we'll first used `VectorAssembler` to place these variables in a column then use `PCA`. When we are ready, we’ll use two fitted PCA features, the binary `Hour`, `Power_Zone_1`, `Power_Zone_2`, and the `Month` indicator variables. 


In [13]:
spower.dtypes

[('Temperature', 'double'),
 ('Humidity', 'double'),
 ('Wind_Speed', 'double'),
 ('General_Diffuse_Flows', 'double'),
 ('Diffuse_Flows', 'double'),
 ('Power_Zone_1', 'double'),
 ('Power_Zone_2', 'double'),
 ('Power_Zone_3', 'double'),
 ('Month', 'bigint'),
 ('Hour', 'bigint')]

In [3]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA

In [46]:
# cast to DoubleType
sqlTrans = SQLTransformer(
    statement="""
    SELECT
        *,
        CAST(Hour AS DOUBLE) AS Hour_dbl,
        `Power_Zone_3` AS label
    FROM __THIS__
    """
)

#Binarize Hour 
binaryTrans = Binarizer(
    threshold=6.5,
    inputCol="Hour_dbl",
    outputCol="Hour_indicator"
)

# create indexer for Month 
month_indexer = StringIndexer(
    inputCol="Month", 
    outputCol="Month_idx", 
    handleInvalid="keep")


# One hot encode Month
month_ohe = OneHotEncoder(
    inputCols=["Month_idx"],
    outputCols=["Month_ohe"],
    dropLast=True
)

#assumble the columns into one column for PCA
pca_input_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="pca_input",
    handleInvalid="keep"
)

# perform PCA
pca = PCA(
    k=2,
    inputCol="pca_input",
    outputCol="pca_features"
)

# assemble feature columns 
feature_assembler = VectorAssembler(
    inputCols=[
        "pca_features",
        "Hour_indicator",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_ohe"
    ],
    outputCol="features",
    handleInvalid="keep"
)



After we define the transformations, we want to fit a Linear regression using CV. We want to build a grid for all combinations of 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1 on `regParam` and `elasticNetParam`. 

In [47]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(featuresCol="features", labelCol="label")
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

Now set up the `Pipeline`. 

In [48]:
#create pipeline 
pipeline = Pipeline(stages=[
    sqlTrans,
    binaryTrans,
    month_indexer,
    month_ohe,
    pca_input_assembler,
    pca,
    feature_assembler, 
    lr
])

Now, we can use the `CrossValidator()` function to run k-fold CV over the grid of tuning parameters we just set up. We'll use the default RMSE metric to evaluate our model. 


In [49]:
crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse"),
    numFolds=5
)

In [ ]:
cvModel = crossval.fit(spower)

26/04/18 18:43:17 WARN CacheManager: Asked to cache already cached data.
26/04/18 18:43:17 WARN CacheManager: Asked to cache already cached data.
26/04/18 18:43:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/18 18:43:18 WARN Instrumentation: [1eae3b5b] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 18:43:19 WARN Instrumentation: [1eae3b5b] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 18:43:21 WARN Instrumentation: [98695792] regParam is zero, which might cause numerical instability and overfitting.
26/04/18 18:43:21 WARN Instrumentation: [98695792] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/18 18:43:23 WARN Instrumentation: [b0f0d116] regParam is zero, which might cause numerical instability and overfitting.

In [ ]:
bestModel = cvModel.bestModel
bestLR = bestModel.stages[-1]

print("Best regParam:", bestLR.getRegParam())
print("Best elasticNetParam:", bestLR.getElasticNetParam())
print("Best CV RMSE:", min(cvModel.avgMetrics))